In [26]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, VotingClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

%load_ext autoreload
%autoreload 2
from preprocess import TitanicPreprocessor

OG_TRAIN_PATH = "../data/original/train.csv"
OG_TEST_PATH = "../data/original/test.csv"
RANDOM_STATE = 42
NUM_FEATURES = ["Age", "SibSp", "Parch", "Fare", "Companion"]
CAT_FEATURES = ["Sex", "Pclass", "Embarked", "hasAge", "hasCabin", "Title", "Deck", "Side", "isAlone", "hasSibSp", "hasParch"]
CAT_NOMINAL_FEATURES = ["Sex", "Title", "Embarked", "isAlone", "hasSibSp", "hasParch", "Side", "hasCabin", "hasAge"]
CAT_ORDINAL_FEATURES = ["Pclass", "Deck"]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 헬퍼 함수


In [27]:
def get_nullcnt_ratio_describe(df: pd.DataFrame) -> pd.DataFrame:
    """주어진 DataFrame 객체에서 결측치가 존재하는 컬럼의 이름과 그 개수, 비율, 그리고 정보를 담은 DataFrame을 리턴합니다."""
    # 각 컬럼별 결측치 개수 계산
    null_counts = df.isnull().sum()
    
    # 결측치가 1개 이상 존재하는 컬럼만 필터링
    null_counts = null_counts[null_counts > 0]
    
    # 결측치 비율 계산 (결측치 개수 / 전체 행 개수)
    null_ratios = null_counts / len(df)

    # 결측치 행의 정보 확인
    null_infos = df[null_counts.index].describe(include='all').transpose()
    
    # 데이터프레임으로 변환
    null_df = pd.DataFrame({
        'Null_Count': null_counts,
        'Null_Ratio': null_ratios,
    })
    null_df = pd.concat([null_df, null_infos], axis=1)
    
    # 결측치 개수를 기준으로 내림차순 정렬하여 반환
    return null_df.sort_values(by='Null_Count', ascending=False)

def classify_num_cat_features(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    """주어진 DataFrame 객체에서 수치형 컬럼과 범주형 컬럼을 분류하여 list의 튜플로 반환합니다.
    NUM_FETURES와 CAT_FEATURES를 기준으로 분류합니다.

    Returns:
        tuple[list[str], list[str]]: (수치형 컬럼 리스트, 범주형 컬럼 리스트)
    """
    num_features = [col for col in df.columns if col in NUM_FEATURES]
    cat_features = [col for col in df.columns if col in CAT_FEATURES]
    return num_features, cat_features

def classify_num_cat_features_with_nomial(df: pd.DataFrame) -> tuple[list[str], list[str], list[str]]:
    """주어진 DataFrame 객체에서 수치형 컬럼, 순서 없는 범주형 컬럼, 순서 있는 범주형 컬럼을 분류하여 list의 튜플로 반환합니다.
    NUM_FETURES, CAT_NOMINAL_FEATURES, CAT_ORDINAL_FEATURES를 기준으로 분류합니다.

    Returns:
        tuple[list[str], list[str], list[str]]: (수치형 컬럼 리스트, 순서 없는 범주형 컬럼 리스트, 순서 있는 범주형 컬럼 리스트)
    """
    num_features = [col for col in df.columns if col in NUM_FEATURES]
    cat_nominal_features = [col for col in df.columns if col in CAT_NOMINAL_FEATURES]
    cat_ordinal_features = [col for col in df.columns if col in CAT_ORDINAL_FEATURES]
    return num_features, cat_nominal_features, cat_ordinal_features


experiment_log = []  # 실험 결과가 쌓이는 곳
def run_experiment(name, pipeline, X_tr, y_tr, X_va, y_va, cv=5):
    """모델을 학습시키고, 성능과 하이퍼파라미터를 experiment_log에 자동으로 기록합니다.

    반환값은 지금까지의 전체 실험 기록을 검증 정확도 순으로 정렬한 DataFrame입니다.
    """
    pipeline.fit(X_tr, y_tr)

    cv_scores = cross_val_score(pipeline, X_tr, y_tr, cv=cv, scoring="accuracy")
    row = {
        "실험명": name,
        "train 정확도": pipeline.score(X_tr, y_tr),
        "검증 정확도": pipeline.score(X_va, y_va),
        "CV 평균": cv_scores.mean(),
        "CV 표준편차": cv_scores.std(),
    }
    row.update(pipeline.named_steps["model"].get_params())
    experiment_log.append(row)

    log_df = pd.DataFrame(experiment_log)
    return log_df.sort_values("검증 정확도", ascending=False).reset_index(drop=True)


def show_log(columns=None):
    """실험 기록을 보여줍니다. columns를 지정하면 그 컬럼들만 골라서 봅니다.

    예: show_log(["실험명", "검증 정확도", "n_estimators", "max_depth"])
    """
    log_df = pd.DataFrame(experiment_log).sort_values("검증 정확도", ascending=False).reset_index(drop=True)
    return log_df[columns] if columns else log_df

## 데이터 전처리
직접 만든 preprocess 모듈을 불러와서 전처리한다. 

In [28]:
preprocessor = TitanicPreprocessor()

train_df, test_df = preprocessor.preprocess(OG_TRAIN_PATH, OG_TEST_PATH)

print("========================================================================")
print("Original train_df shape:", preprocessor.og_train_df.shape)
print("Original train_df columns:", preprocessor.og_train_df.columns)
print("------------------------------------------------------------------------")
print("Preprocessed train_df shape:", train_df.shape)
print("Preprocessed train_df columns:", train_df.columns)
print("------------------------------------------------------------------------")
print("Added columns:", set(train_df.columns) - set(preprocessor.og_train_df.columns))
print("========================================================================")
print("Original test_df shape:", preprocessor.og_test_df.shape)
print("Original test_df columns:", preprocessor.og_test_df.columns)
print("------------------------------------------------------------------------")
print("Preprocessed test_df shape:", test_df.shape)
print("Preprocessed test_df columns:", test_df.columns)
print("------------------------------------------------------------------------")
print("Added columns:", set(test_df.columns) - set(preprocessor.og_test_df.columns))
print("========================================================================")

print("Preprocessed train_df head:")
display(train_df.head(3))
print("Preprocessed test_df head:")
display(test_df.head(3))

Original train_df shape: (891, 12)
Original train_df columns: Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')
------------------------------------------------------------------------
Preprocessed train_df shape: (891, 21)
Preprocessed train_df columns: Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'hasAge', 'hasCabin',
       'Title', 'Deck', 'Side', 'Companion', 'isAlone', 'hasSibSp',
       'hasParch'],
      dtype='str')
------------------------------------------------------------------------
Added columns: {'hasAge', 'Deck', 'hasParch', 'Companion', 'Side', 'hasCabin', 'hasSibSp', 'isAlone', 'Title'}
Original test_df shape: (418, 11)
Original test_df columns: Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')
----

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,Embarked,hasAge,hasCabin,Title,Deck,Side,Companion,isAlone,hasSibSp,hasParch
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,...,S,1,0,Mr,Unknown,Unknown,1,0,1,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,...,C,1,1,Mrs,C,Starboard,1,0,1,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,...,S,1,0,Miss,Unknown,Unknown,0,1,0,0


Preprocessed test_df head:


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,hasAge,hasCabin,Title,Deck,Side,Companion,isAlone,hasSibSp,hasParch
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q,1,0,Mr,Unknown,Unknown,0,1,0,0
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S,1,0,Mrs,Unknown,Unknown,1,0,1,0
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q,1,0,Mr,Unknown,Unknown,0,1,0,0


In [29]:
print("train_df 결측치 정보:")
display(get_nullcnt_ratio_describe(train_df))
print("test_df 결측치 정보:")
display(get_nullcnt_ratio_describe(test_df))

train_df 결측치 정보:


,Null_Count,Null_Ratio,count,unique,top,freq
Cabin,687,0.771044,204,147,G6,4


test_df 결측치 정보:


,Null_Count,Null_Ratio,count,unique,top,freq
Cabin,327,0.782297,91,76,B57 B59 B63 B66,3


In [30]:
print("preprocessed train_df info:")
display(train_df.info())

preprocessed train_df info:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 21 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     891 non-null    str    
 12  hasAge       891 non-null    int64  
 13  hasCabin     891 non-null    int64  
 14  Title        891 non-null    str    
 15  Deck         891 non-null    str    
 16  Side         891 non-null    str    
 17  Companion    891 non-null    int64  
 18  isAlone      891 non-null    int6

None

## 데이터 분할 

In [31]:
# 훈련에 사용하지 않을 컬럼은 제거한다.
# 동승자 정보를 표현하는 컬럼이 여러 개 존재하므로, 어떤 컬럼을 제거했을 때 가장 성능이 좋을 지 실험한다. 
base = ["PassengerId", "Name", "Ticket", "Cabin"]
columns_to_drop = {
    # 다중공선성 고려를 하지 않음
    "base": base,
    # isAlone, hasSibSp, hasParch 컬럼만 사용
    "only_flag": base + ["SibSp", "Parch", "Companion"], 
    # isAlone 컬럼만 사용
    "only_isAlone": base + ["SibSp", "Parch", "Companion", "hasSibSp", "hasParch"],
    # hasSibSp, hasParch 컬럼만 사용
    "only_hasSibSp_hasParch": base + ["SibSp", "Parch", "Companion", "isAlone"],
    # Companion 컬럼만 사용 
    "only_Companion": base + ["SibSp", "Parch", "isAlone", "hasSibSp", "hasParch"],
    # SibSp, Parch 컬럼만 사용
    "only_SibSp_Parch": base + ["Companion", "isAlone", "hasSibSp", "hasParch"]
}

# columns_to_drop에 정의된 컬럼들을 제거한 DataFrame을 생성한다.
droped_train_df_dict = {key: train_df.drop(columns=cols) for key, cols in columns_to_drop.items()}
droped_test_df_dict = {key: test_df.drop(columns=cols) for key, cols in columns_to_drop.items()}

for train, test in zip(droped_train_df_dict.items(), droped_test_df_dict.items()):
    print(f"TRAIN DataFrame '{train[0]}' shape: {train[1].shape}")
    print(train[1].columns.tolist())
    print(f"TEST DataFrame '{test[0]}' shape: {test[1].shape}")
    print(test[1].columns.tolist())
    print("========================================================================")


TRAIN DataFrame 'base' shape: (891, 17)
['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'hasAge', 'hasCabin', 'Title', 'Deck', 'Side', 'Companion', 'isAlone', 'hasSibSp', 'hasParch']
TEST DataFrame 'base' shape: (418, 16)
['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'hasAge', 'hasCabin', 'Title', 'Deck', 'Side', 'Companion', 'isAlone', 'hasSibSp', 'hasParch']
TRAIN DataFrame 'only_flag' shape: (891, 14)
['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'hasAge', 'hasCabin', 'Title', 'Deck', 'Side', 'isAlone', 'hasSibSp', 'hasParch']
TEST DataFrame 'only_flag' shape: (418, 13)
['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'hasAge', 'hasCabin', 'Title', 'Deck', 'Side', 'isAlone', 'hasSibSp', 'hasParch']
TRAIN DataFrame 'only_isAlone' shape: (891, 12)
['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'hasAge', 'hasCabin', 'Title', 'Deck', 'Side', 'isAlone']
TEST DataFrame 'only_isAlone' shape: (418, 11)
['Pclass', 'Sex', 'Age', 'Fa

## 베이스라인 모델 찾기
이번 실습에서 실험할 모델은 Logistic Regression, KNN, Decision Tree이다. 그리고 이 세 모델을 조합해서 앙상블까지 실험한다. 

세 모델은 각각 특성이 다르기 때문에, 전처리기 구성도 다르게 가져가야 한다.
- **Ecoding**
    | 방법 | 설명 | 언제 쓰나 | 주의점 |
    |------|------|----------|--------|
    | **레이블 인코딩** | 각 범주에 0, 1, 2 같은 번호 부여 | (주로) **타깃 y** 라벨, 또는 **순서형**을 임시로 코드화 | X에 쓰면 **가짜 순서** 위험 (특히 선형/거리 모델) |
    | **원-핫 인코딩** | red=[1,0,0], green=[0,1,0] | **순서 없는 명목형 X** (순서를 만들지 않는 출발점) | 차원 증가(카테고리 많으면 폭발) |
    | **오디널 인코딩** | small=0, medium=1, large=2 | **순서가 의미 있을 때** | 순서를 직접 지정해야 함 |
- **Scaling**
    | 스케일러 | 방식 | 특징 | 언제 쓰나 |
    |---------|------|------|----------|
    | **StandardScaler** | 평균=0, 표준편차=1 (z-score) | 많은 모델에서 안정적으로 잘 작동 | 기본 선택(선형/거리 기반에서 자주) |
    | **MinMaxScaler** | [0, 1] 구간으로 변환 | 이상치(outlier)에 민감 | 범위가 중요한 값(픽셀 등) / 데이터가 깔끔할 때 |
    | **RobustScaler** | 중앙값/IQR 기반 | 이상치에 강함 | 이상치가 있을 때 |

- **Normalization**
    - 정규화의 목적: **길이 영향은 지우고, 방향(패턴)만 비교**

### Logistic Regression, KNN
- **Encoding**: One-Hot Encoding. 고정. 
    - Ordinal Encoding을 쓰면, category 간에 선형적인 관계를 강제로 부여하게 된다. 우리가 가지고 있는 Feature 중에서는 선형적인 관계를 가지고 있는 Feature는 없다.     
- **Scaling**: 세 가지의 스케일링 알고리즘을 실험해 하나를 선택한다. 

### Decision Tree
- **Encoding**: One-Hot, Ordinal 두 개를 실험으로 선택한다. 
- **Scaling**: DecisionTree는 스케일링이 전혀 의미 없다. 따라서 아예 적용하지 않는다. 

In [32]:
experiment_results=[]

scoring_metrics = ['accuracy', 'precision', 'recall', 'f1']

# 데이터 조합에 대해서 파이프라인(전처리+모델) 조합 찾기. 하이퍼파라미터 튜닝 없음. 
for data_key, df in droped_train_df_dict.items():
    print("[INFO] 테스트 중인 데이터: ", data_key)

    # 훈련 데이터 준비
    y_train = df["Survived"]
    X_train = df.drop(columns=["Survived"])

    # 현재 데이터셋에 대해 수치형, 순서 없는 범주형, 순서 있는 범주형 컬럼 분류
    num_features, cat_nominal_features, cat_ordinal_features = classify_num_cat_features_with_nomial(X_train)

    # 더미 전처리기. GridSearchCV로 모델별로 전처리기를 다르게 테스트한다. 
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_features),
            ('nom', OneHotEncoder(handle_unknown='ignore'), cat_nominal_features),
            ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_ordinal_features)
        ],
        remainder='drop'
    )

    # Pipeline 뼈대 생성.
    pipeline = Pipeline(
        steps=[
            ('prep', preprocessor),
            ('model', LogisticRegression(random_state=RANDOM_STATE))
        ]
    )

    param_grid = [
        {
            # [실험군 1] Logistic Regression (선형 기반)
            'model': [LogisticRegression(max_iter=1000, random_state=42)],
            
            # 스케일러 3종을 교체하며 실험
            'prep__num': [StandardScaler(), MinMaxScaler(), RobustScaler()], 
            
            # LR은 Ordinal을 쓰면 안 되므로, 'ord' 단계의 변환기를 OneHotEncoder로 덮어씌움
            'prep__ord': [OneHotEncoder(handle_unknown='ignore')] 
        },
        {
            # [실험군 2] KNN (거리 기반)
            'model': [KNeighborsClassifier(n_neighbors=5)],
            
            # 스케일러 3종을 교체하며 실험
            'prep__num': [StandardScaler(), MinMaxScaler(), RobustScaler()], 
            
            # KNN은 Ordinal을 쓰면 안 되므로, 'ord' 단계의 변환기를 OneHotEncoder로 덮어씌움
            'prep__ord': [OneHotEncoder(handle_unknown='ignore')] 
        },
        {
            # [실험군 3] Decision Tree (트리 기반)
            'model': [DecisionTreeClassifier(random_state=42)],
            
            # 트리는 스케일링이 의미 없으므로 'passthrough'로 건너뜀
            'prep__num': ['passthrough'], 
            
            # 'prep__ord' 파라미터를 안 적었으므로 더미에 있던 OrdinalEncoder가 그대로 작동함
            
            # 간단한 베이스라인 하이퍼파라미터 탐색
            'model__max_depth': [3, 5, 7, 10] 
        }
    ]

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
        scoring=scoring_metrics, # 불균형 데이터이므로, recall, precision, f1 등 다양한 지표를 확인
        refit='accuracy', # 최종 결정은 accuracy를 기준으로 
        n_jobs=-1 # 가능한 모든 CPU 코어를 동원
    )
    grid_search.fit(X_train, y_train)

    cv_results = grid_search.cv_results_

    for i, params in enumerate(cv_results['params']):
        
        model_obj = params.get('model')
        model_name = model_obj.__class__.__name__
        
        prep_num = params.get('prep__num')
        scaler_name = 'passthrough' if prep_num == 'passthrough' else prep_num.__class__.__name__
        
        prep_ord = params.get('prep__ord', 'OrdinalEncoder (Default)')
        encoder_name = prep_ord if isinstance(prep_ord, str) else prep_ord.__class__.__name__

        experiment_results.append({
            'Feature Set': data_key,
            'Model': model_name,
            'Scaler': scaler_name,
            'Encoder': encoder_name,
            'Accuracy': cv_results['mean_test_accuracy'][i],
            'Precision': cv_results['mean_test_precision'][i],
            'Recall': cv_results['mean_test_recall'][i],
            'F1-Score': cv_results['mean_test_f1'][i]
        })

all_results_df = pd.DataFrame(experiment_results)

best_idx_per_group = all_results_df.groupby(['Feature Set', 'Model'])['Accuracy'].idxmax()

final_summary_df = all_results_df.loc[best_idx_per_group].sort_values(
    by=['Feature Set', 'Accuracy'], 
    ascending=[True, False]
)

print("\n=== 데이터셋 & 모델별 최적 전처리 조합 ===")
display(final_summary_df)

[INFO] 테스트 중인 데이터:  base
[INFO] 테스트 중인 데이터:  only_flag
[INFO] 테스트 중인 데이터:  only_isAlone
[INFO] 테스트 중인 데이터:  only_hasSibSp_hasParch
[INFO] 테스트 중인 데이터:  only_Companion
[INFO] 테스트 중인 데이터:  only_SibSp_Parch

=== 데이터셋 & 모델별 최적 전처리 조합 ===


,Feature Set,Model,Scaler,Encoder,Accuracy,Precision,Recall,F1-Score
0,base,LogisticRegression,StandardScaler,OneHotEncoder,0.826044,0.781104,0.762830,0.770009
6,base,DecisionTreeClassifier,passthrough,OrdinalEncoder (Default),0.818153,0.794636,0.724808,0.749787
4,base,KNeighborsClassifier,MinMaxScaler,OneHotEncoder,0.797998,0.766158,0.684314,0.720631
41,only_Companion,LogisticRegression,MinMaxScaler,OneHotEncoder,0.829396,0.791479,0.757033,0.772254
46,only_Companion,DecisionTreeClassifier,passthrough,OrdinalEncoder (Default),0.818153,0.794636,0.724808,0.749787
45,only_Companion,KNeighborsClassifier,RobustScaler,OneHotEncoder,0.804745,0.765364,0.710571,0.735945
51,only_SibSp_Parch,LogisticRegression,MinMaxScaler,OneHotEncoder,0.829402,0.791560,0.757076,0.772332
55,only_SibSp_Parch,KNeighborsClassifier,RobustScaler,OneHotEncoder,0.822698,0.788592,0.736871,0.760583
56,only_SibSp_Parch,DecisionTreeClassifier,passthrough,OrdinalEncoder (Default),0.820400,0.799241,0.721867,0.751138
16,only_flag,DecisionTreeClassifier,passthrough,OrdinalEncoder (Default),0.820400,0.799241,0.721867,0.751138


## GridSearchCV로 하이퍼파라미터 찾기
위 베이스라인 선정에서 각각의 Feature Set에서, LR, KNN, DT에게 가장 궁합이 잘 맞는 스케일러, 인코더 조합을 얻었다.

이제 LR, KNN, DT, RF, GB, Bagging 6가지 모델에 각각의 스케일러, 인코더 조합을 적용하고, 하이퍼 파라미터를 튜닝한다. 

- 각 Feature Set에 대해서
    1. 모델 별로
        - 정해진 스케일러, 인코더 조합을 넣고
        - 하이퍼 파라미터를 튜닝한다
        최고 성능의 모델과 그 하이퍼 파라미터를 기록한다.

    2. 최적의 하이퍼파라미터로 튜닝된 6개 모델에 대해서 soft Voting을 돌린다. 

In [33]:
# 위의 결과에서 각 데이터셋과 모델별로 가장 높은 정확도(Accuracy)를 기록한 스케일러 조합 구하기
scaler_objects = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

# 2. 로지스틱 회귀 모델의 결과만 추출
lr_best_results = final_summary_df[final_summary_df['Model'] == 'LogisticRegression']

# 3. 데이터셋별 최적 스케일러를 딕셔너리로 저장
lr_scaler_map = {}
for _, row in lr_best_results.iterrows():
    feature_set = row['Feature Set']
    scaler_str = row['Scaler']
    
    # 헬퍼 딕셔너리를 통해 실제 객체 할당
    lr_scaler_map[feature_set] = scaler_objects[scaler_str]

print("\n[INFO] 동적으로 생성된 lr_scaler_map:")
for k, v in lr_scaler_map.items():
    print(f" - {k}: {v.__class__.__name__}")


[INFO] 동적으로 생성된 lr_scaler_map:
 - base: StandardScaler
 - only_Companion: MinMaxScaler
 - only_SibSp_Parch: MinMaxScaler
 - only_flag: StandardScaler
 - only_hasSibSp_hasParch: StandardScaler
 - only_isAlone: StandardScaler


In [34]:
final_ensemble_results = []
final_models = []

for data_key, df in droped_train_df_dict.items():
    print(f"\n[INFO] 테스트 중인 데이터셋 '{data_key}'")

    # 훈련 데이터 준비
    y_train = df["Survived"]
    X_train = df.drop(columns=["Survived"])

    # 현재 데이터셋에 대해 수치형, 순서 없는 범주형, 순서 있는 범주형 컬럼 분류
    num_features, cat_nominal_features, cat_ordinal_features = classify_num_cat_features_with_nomial(X_train)
    
    # LR용 전처리기: 데이터셋 이름에 따라 스케일러가 동적으로 바뀜
    prep_lr = ColumnTransformer([
        ('num', lr_scaler_map.get(data_key, StandardScaler()), num_features),
        ('nom', OneHotEncoder(handle_unknown='ignore'), cat_nominal_features),
        ('ord', OneHotEncoder(handle_unknown='ignore'), cat_ordinal_features) 
    ], remainder='drop')
    
    # KNN용 전처리기: 항상 RobustScaler 선호
    prep_knn = ColumnTransformer([
        ('num', RobustScaler(), num_features),
        ('nom', OneHotEncoder(handle_unknown='ignore'), cat_nominal_features),
        ('ord', OneHotEncoder(handle_unknown='ignore'), cat_ordinal_features) 
    ], remainder='drop')
    
    # 트리용 전처리기: 항상 스케일링 패스 + OrdinalEncoder
    prep_tree = ColumnTransformer([
        ('num', 'passthrough', num_features),
        ('nom', OneHotEncoder(handle_unknown='ignore'), cat_nominal_features),
        ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_ordinal_features)
    ], remainder='drop')

    # Phase 1: 6개 모델 독립 튜닝
    # TODO: 각 하이퍼 파라미터 후보 조절 가능
    model_configs = {
        'LR': (
            Pipeline([('prep', prep_lr), ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
            {'model__C': [0.1, 1, 10]}
        ),
        'KNN': (
            Pipeline([('prep', prep_knn), ('model', KNeighborsClassifier())]),
            {'model__n_neighbors': [3, 5, 7], 'model__weights': ['uniform', 'distance']}
        ),
        'DT': (
            Pipeline([('prep', prep_tree), ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))]),
            {'model__max_depth': [3, 5, 7, None]}
        ),
        'RF': (
            Pipeline([('prep', prep_tree), ('model', RandomForestClassifier(random_state=RANDOM_STATE))]),
            {'model__n_estimators': [100, 200], 'model__max_depth': [5, 7, 10]}
        ),
        'GB': (
            Pipeline([('prep', prep_tree), ('model', GradientBoostingClassifier(random_state=RANDOM_STATE))]),
            {'model__learning_rate': [0.05, 0.1], 'model__n_estimators': [100, 200], 'model__max_depth': [3, 5]}
        ),
        'Bagging': (
            Pipeline([('prep', prep_tree), ('model', BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE), random_state=RANDOM_STATE))]),
            {'model__n_estimators': [50, 100], 'model__max_samples': [0.7, 1.0]}
        )
    }

    best_estimators = {} 

    print("[INFO] 개별 모델 하이퍼파라미터 튜닝 중...")
    for name, (pipe, params) in model_configs.items():
        search = GridSearchCV(pipe, params, cv=5, scoring='accuracy', n_jobs=-1)
        search.fit(X_train, y_train)
        best_estimators[name] = search.best_estimator_ 
        
    # Phase 2: 최종 앙상블 (Voting)
    print("[INFO] Voting 앙상블 결합 및 가중치 튜닝 중...")
    
    voting_clf = VotingClassifier(
        estimators=[(name, est) for name, est in best_estimators.items()],
        voting='soft' 
    )

    # TODO: 가중치 조절 가능
    voting_params = {
        'weights': [
            [1, 1, 1, 1, 1, 1], # 균등 투표
            [2, 1, 1, 2, 2, 1], # LR, RF, GB에 가중치 2배
            [1, 1, 1, 2, 2, 2]  # 앙상블 트리 모델들(RF, GB, Bagging)에 가중치 2배
        ]
    }
    
    voting_search = GridSearchCV(voting_clf, voting_params, cv=5, scoring='accuracy', n_jobs=-1)
    voting_search.fit(X_train, y_train)
    
    final_ensemble_results.append({
        'Feature Set': data_key,
        'Voting Accuracy': voting_search.best_score_,
        'Best Weights': voting_search.best_params_['weights']
    })

    final_models.append({
        'Feature Set': data_key,
        'Model': voting_search
    })

final_df = pd.DataFrame(final_ensemble_results).sort_values(by='Voting Accuracy', ascending=False)
print("[INFO] 최종 앙상블(Voting) 딥 튜닝 결과 ===")
display(final_df)


[INFO] 테스트 중인 데이터셋 'base'
[INFO] 개별 모델 하이퍼파라미터 튜닝 중...
[INFO] Voting 앙상블 결합 및 가중치 튜닝 중...

[INFO] 테스트 중인 데이터셋 'only_flag'
[INFO] 개별 모델 하이퍼파라미터 튜닝 중...
[INFO] Voting 앙상블 결합 및 가중치 튜닝 중...

[INFO] 테스트 중인 데이터셋 'only_isAlone'
[INFO] 개별 모델 하이퍼파라미터 튜닝 중...
[INFO] Voting 앙상블 결합 및 가중치 튜닝 중...

[INFO] 테스트 중인 데이터셋 'only_hasSibSp_hasParch'
[INFO] 개별 모델 하이퍼파라미터 튜닝 중...
[INFO] Voting 앙상블 결합 및 가중치 튜닝 중...

[INFO] 테스트 중인 데이터셋 'only_Companion'
[INFO] 개별 모델 하이퍼파라미터 튜닝 중...
[INFO] Voting 앙상블 결합 및 가중치 튜닝 중...

[INFO] 테스트 중인 데이터셋 'only_SibSp_Parch'
[INFO] 개별 모델 하이퍼파라미터 튜닝 중...
[INFO] Voting 앙상블 결합 및 가중치 튜닝 중...
[INFO] 최종 앙상블(Voting) 딥 튜닝 결과 ===


,Feature Set,Voting Accuracy,Best Weights
5,only_SibSp_Parch,0.833896,"[2, 1, 1, 2, 2, 1]"
4,only_Companion,0.831655,"[2, 1, 1, 2, 2, 1]"
3,only_hasSibSp_hasParch,0.830538,"[2, 1, 1, 2, 2, 1]"
1,only_flag,0.827167,"[2, 1, 1, 2, 2, 1]"
2,only_isAlone,0.827161,"[1, 1, 1, 1, 1, 1]"
0,base,0.826050,"[2, 1, 1, 2, 2, 1]"


## 최종 모델로 테스트 데이터를 예측하기

In [ ]:
for item in final_models:
    data_key = item['Feature Set']
    model = item['Model']
    
    # 이 모델에 맞는 형태의 테스트 데이터 꺼내기
    X_test = droped_test_df_dict[data_key]
    
    # 저장해둔 모델로 예측 수행
    predictions = model.predict(X_test)

    # 제출용 DataFrame 생성
    # PassengerId는 drop 과정에서 날아갔으므로 원본 test_df에서 가져옴
    submission = pd.DataFrame({
        'PassengerId': test_df['PassengerId'],
        'Survived': predictions
    })
    
    # 파일로 저장
    file_name = f'submission_{data_key}.csv'
    submission.to_csv(f'../data/prediction/{file_name}', index=False)
    print(f"{file_name} 저장 완료")

submission_base.csv 저장 완료
submission_only_flag.csv 저장 완료
submission_only_isAlone.csv 저장 완료
submission_only_hasSibSp_hasParch.csv 저장 완료
submission_only_Companion.csv 저장 완료
submission_only_SibSp_Parch.csv 저장 완료


## 휴지통


In [36]:
# 모델과 하이퍼파라미터 그리드 후보
pass

model_grids = {
    'Logistic Regression': (
        LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
        [
            # 조합 1: l2 규제는 lbfgs, liblinear 모두 테스트
            {
                'model__penalty': ['l2'], 
                'model__solver': ['lbfgs', 'liblinear'],
                'model__C': [0.01, 0.1, 1, 10]
            },
            # 조합 2: l1 규제는 지원하는 liblinear, saga로만 테스트
            {
                'model__penalty': ['l1'], 
                'model__solver': ['liblinear', 'saga'],
                'model__C': [0.01, 0.1, 1, 10]
            }
        ]
    ),
    'KNN': (
        KNeighborsClassifier(),
        {
            'model__n_neighbors': [3, 5, 7, 9, 11],
            'model__weights': ['uniform', 'distance'],
            'model__metric': ['euclidean', 'manhattan', 'minkowski'],
            'model__p': [1, 2]
        }
    ),
    'Decision Tree': (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {
            'model__max_depth': [3, 5, 7, 10, 20],
            'model__min_samples_split': [2, 5, 10, 20],
            'model__min_samples_leaf': [1, 2, 4, 8, 15],
            'model__max_leaf_nodes': [None, 5, 10, 20, 50]
        }
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=RANDOM_STATE),
        {
            'model__n_estimators': [100, 200, 300],
            'model__max_depth': [3, 5, 7, 10, None],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4]
        }
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            'model__n_estimators': [100, 200, 300],
            'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'model__max_depth': [3, 5, 7],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4]
        }
    ),
    'Bagging': (
        BaggingClassifier(random_state=RANDOM_STATE),
        {
            'model__n_estimators': [10, 50, 100],
            'model__max_samples': [0.5, 0.7, 1.0],
            'model__max_features': [0.5, 0.7, 1.0],
            'model__bootstrap': [True, False],
            'model__bootstrap_features': [True, False],
            'model__estimator__max_depth': [3, 5, 7, None]
        }
    ),
    'Voting': (
        VotingClassifier(
            estimators=[
                ('lr', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)),
                ('knn', KNeighborsClassifier()),
                ('dt', DecisionTreeClassifier(random_state=RANDOM_STATE)),
                ('rf', RandomForestClassifier(random_state=RANDOM_STATE)),
                ('gb', GradientBoostingClassifier(random_state=RANDOM_STATE))
            ],
            voting='soft'
        ),
        {
            'model__weights': [[1, 1, 1, 1, 1], [2, 1, 1, 1, 1], [1, 2, 1, 1, 1], [1, 1, 2, 1, 1], [1, 1, 1, 2, 1], [1, 1, 1, 1, 2]],
            'model__voting': ['hard', 'soft']
        }
    )
}

## RandomSearchCV로 하이퍼파라미터 찾기